In [ ]:
# retriever in langchain
# i hae use google colab
# !pip install langchain langchain-community pypdf chromadb langchain-text-splitters Transformers langchain_groq wikipedia

In [19]:
from langchain_community.document_loaders import WikipediaLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.retrievers import WikipediaRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
from google.colab import userdata
userdata.get('Groq_Key')
userdata.get('HF_TOKEN')



In [ ]:
# use wikipedia retriver(searching) not loader
# docs_wiki_retriver=WikipediaRetriever()    
# docs_wiki_retriver.invoke("top 3 ai researcher in world with only name")

In [31]:
pdf_loader=PyPDFLoader('/content/DAA.pdf')
pdf_docs=pdf_loader.load()


In [5]:
#   my workflow  1)load docs---> 2)split it ----> 3)Genrate embedding --->4) store it --->
# --> use retrivert to fetch data
loader = WikipediaLoader(query="Generative AI", load_max_docs=3)
wiki_docs = loader.load()

In [ ]:
docs_chunk=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(pdf_docs)   #use your target documents
len(docs_chunk)
docs_chunk[2].page_content



'Copyright © 2012, 2007, 2003 Pearson Education, Inc., publishing as Addison-Wesley. All rights\nreserved. Printed in the United States of America. This publication is protected by Copyright,\nand permission should be obtained from the publisher prior to any prohibited reproduction,\nstorage in a retrieval system, or transmission in any form or by any means, electronic, mechanical,\nphotocopying, recording, or likewise. T o obtain permission(s) to use material from this work,\nplease submit a written request to Pearson Education, Inc., Permissions Department, One Lake\nStreet, Upper Saddle River, New Jersey 07458, or you may fax your request to 201-236-3290.\nThis is the eBook of the printed book and may not include any media, Website access codes or\nprint supplements that may come packaged with the bound book.\nMany of the designations by manufacturers and sellers to distinguish their products are claimed'

In [50]:
# genrate embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={},
    encode_kwargs={},
    # query_instruction="Represent the query for retrieval: " # Removed as it's an extra input
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
vector_store=Chroma.from_documents(
    documents=docs_chunk,
    embedding=embeddings,
    collection_name="my_new_collection" # Changed collection name for a fresh start
    )

In [ ]:
retriver=vector_store.as_retriever(search_kwargs={"k":2})   #---   retriver is runnable


In [ ]:
query = "Explain greedy algorithm  in detail"
result = retriver.invoke(query)
for doc in result:
  print(doc.page_content)